In [1]:
"""
Preprocess and scale CIC_ToN_IoT (target domain). Reuse scaler from CICIDS2017,
and calculate covariance statistics.
"""

### Imports ###
import json
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

# Load shared feature-space artifacts in a single, validated format.
def load_feature_order(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict):
        if "features" not in payload:
            raise ValueError(
                f"Expected key 'features' in {path} when JSON object is provided."
            )
        feature_order = list(payload["features"])
    elif isinstance(payload, list):
        feature_order = list(payload)
    else:
        raise ValueError(
            f"Unsupported shared feature space format in {path}: "
            f"{type(payload).__name__}"
        )

    if not feature_order:
        raise ValueError(f"Shared feature space in {path} is empty")

    return feature_order

In [2]:
### Import CSV ###

# Creates a Path object pointing to the target-domain CSV directory.
data_dir = Path("data/raw/target")

# Read CSV with encoding fallback for files that are not UTF-8.
def read_csv_with_fallback(file_path):
    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(file_path, low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("unknown", b"", 0, 1, f"Unable to decode {file_path}")

# Target domain is expected to be a single CSV file.
csv_files = sorted(data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {data_dir}")
if len(csv_files) != 1:
    raise ValueError(
        f"Expected exactly one target CSV in {data_dir}, found {len(csv_files)}: "
        f"{[p.name for p in csv_files]}"
    )

csv_path = csv_files[0]
df = read_csv_with_fallback(csv_path)

# Display a quick shape check and preview rows.
print(f"Loaded target CSV: {csv_path.name}")
print("Dataset shape:", df.shape)
df.head()

Loaded target CSV: CIC-ToN-IoT.csv
Dataset shape: (5351760, 85)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Attack
0,177.30.87.144-192.168.1.1-0-0-0,177.30.87.144,0,192.168.1.1,0,0,25/04/2019 05:18:52 pm,47814343,5,0,...,1038036.0,0.000000e+00,1038036.0,1038036.0,5.187256e+14,8.984590e+14,1.556177e+15,1.657324e+07,0,Benign
1,167.49.176.28-50.165.192.168-0-0-0,167.49.176.28,0,50.165.192.168,0,0,25/04/2019 05:18:49 pm,2033142,2,0,...,0.0,0.000000e+00,0.0,0.0,1.556177e+15,0.000000e+00,1.556177e+15,1.556177e+15,0,Benign
2,230.158.52.59-177.21.192.168-0-0-0,230.158.52.59,0,177.21.192.168,0,0,25/04/2019 05:18:37 pm,82877133,14,0,...,1931160.5,1.711593e+06,3942470.0,226402.0,1.729085e+14,5.187256e+14,1.556177e+15,6.036493e+06,0,Benign
3,183.68.192.168-1.1.192.168-0-0-0,183.68.192.168,0,1.1.192.168,0,0,25/04/2019 05:18:42 pm,24359,2,0,...,0.0,0.000000e+00,0.0,0.0,1.556177e+15,0.000000e+00,1.556177e+15,1.556177e+15,0,Benign
4,183.41.192.168-1.1.192.168-0-0-0,183.41.192.168,0,1.1.192.168,0,0,25/04/2019 05:18:42 pm,10239351,3,0,...,4053975.0,0.000000e+00,4053975.0,4053975.0,7.780884e+14,1.100383e+15,1.556177e+15,6.185376e+06,0,Benign


In [3]:
### Data sanitization ###

# Handle missing values by replacing all occurrences of infinity with NaN
# then removing rows containing NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Remove duplicate flows and irrelevant columns.
df.drop_duplicates(inplace=True)
df = df.drop(columns=["Flow ID", "Src IP", "Dst IP", "Timestamp"], errors="ignore")

# Remove leading/trailing spaces from all column names.
df.rename(columns=lambda x: x.strip(), inplace=True)

# Target-label correction: this dataset's semantic labels are in 'Attack'.
# 1) Remove the binary 'Label' column if present.
# 2) Rename 'Attack' to canonical 'Label' for downstream alignment/encoding.
if "Attack" not in df.columns:
    raise ValueError("Expected 'Attack' column in target dataset but it was not found.")

if "Label" in df.columns:
    df.drop(columns=["Label"], inplace=True)

df.rename(columns={"Attack": "Label"}, inplace=True)
print("Target label transformation applied: dropped original 'Label' and renamed 'Attack' -> 'Label'.")

# Display a quick shape check and preview rows.
print(f"Loaded target CSV: {csv_path.name}")
print("Dataset shape:", df.shape)
df.head()

Target label transformation applied: dropped original 'Label' and renamed 'Attack' -> 'Label'.
Loaded target CSV: CIC-ToN-IoT.csv
Dataset shape: (5350583, 80)


,Src Port,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0,0,0,47814343,5,0,0.0,0.0,0.0,0.0,...,0,1038036.0,0.000000e+00,1038036.0,1038036.0,5.187256e+14,8.984590e+14,1.556177e+15,1.657324e+07,Benign
1,0,0,0,2033142,2,0,0.0,0.0,0.0,0.0,...,0,0.0,0.000000e+00,0.0,0.0,1.556177e+15,0.000000e+00,1.556177e+15,1.556177e+15,Benign
2,0,0,0,82877133,14,0,0.0,0.0,0.0,0.0,...,0,1931160.5,1.711593e+06,3942470.0,226402.0,1.729085e+14,5.187256e+14,1.556177e+15,6.036493e+06,Benign
3,0,0,0,24359,2,0,0.0,0.0,0.0,0.0,...,0,0.0,0.000000e+00,0.0,0.0,1.556177e+15,0.000000e+00,1.556177e+15,1.556177e+15,Benign
4,0,0,0,10239351,3,0,0.0,0.0,0.0,0.0,...,0,4053975.0,0.000000e+00,4053975.0,4053975.0,7.780884e+14,1.100383e+15,1.556177e+15,6.185376e+06,Benign


In [4]:
### Feature-space alignment ###
# (align features according to predetermined shared feature space)

# Canonical feature-space contract shared by source and target pipelines.
FEATURE_LIST_PATH = Path("data/processed/shared_feature_space.json")
FEATURE_ALIAS_MAP_PATH = Path("data/processed/target_feature_alias_map.json")

# Handle missing files early.
if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(
        f"Shared feature list not found at {FEATURE_LIST_PATH}. "
        "Create/populate this artifact before running preprocessing."
    )
if not FEATURE_ALIAS_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Target feature alias map not found at {FEATURE_ALIAS_MAP_PATH}. "
        "Create/populate this artifact before running preprocessing."
    )

# Load canonical ordered features used by source preprocessing/model training.
shared_features = load_feature_order(FEATURE_LIST_PATH)

with open(FEATURE_ALIAS_MAP_PATH, "r", encoding="utf-8") as f:
    target_feature_alias_map = json.load(f)
if not isinstance(target_feature_alias_map, dict):
    raise ValueError(
        f"Expected JSON object at {FEATURE_ALIAS_MAP_PATH}, got "
        f"{type(target_feature_alias_map).__name__}"
    )

# Ensure mapping keys match canonical feature list exactly.
alias_keys = set(target_feature_alias_map.keys())
expected_keys = set(shared_features)
missing_alias_keys = sorted(expected_keys - alias_keys)
extra_alias_keys = sorted(alias_keys - expected_keys)
if missing_alias_keys or extra_alias_keys:
    raise ValueError(
        f"Alias-map key mismatch. Missing keys: {missing_alias_keys[:10]} "
        f"(total={len(missing_alias_keys)}); extra keys: {extra_alias_keys[:10]} "
        f"(total={len(extra_alias_keys)}). Update {FEATURE_ALIAS_MAP_PATH}."
    )

# Label column can vary slightly by export; detect robustly.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)

# Ensure contiguous row index before splitting/rejoining feature and label columns.
df = df.reset_index(drop=True)
feature_df = df.drop(columns=[label_col]).copy() if label_col else df.copy()

# Build canonical feature frame using external canonical->raw mapping.
resolved_raw_by_canonical = {}
missing_raw_columns = []
for canonical in shared_features:
    raw_name = target_feature_alias_map.get(canonical, canonical)
    if not isinstance(raw_name, str):
        raise ValueError(
            f"Alias value for canonical feature '{canonical}' must be a string, "
            f"got {type(raw_name).__name__}"
        )
    raw_name = raw_name.strip() or canonical
    resolved_raw_by_canonical[canonical] = raw_name
    if raw_name not in feature_df.columns:
        missing_raw_columns.append((canonical, raw_name))

if missing_raw_columns:
    preview = missing_raw_columns[:10]
    raise ValueError(
        f"Mapped raw target columns not found: {preview} "
        f"(total={len(missing_raw_columns)}). Update {FEATURE_ALIAS_MAP_PATH}."
    )

# Supports one raw column feeding multiple canonical features (e.g., Fwd Header Len).
aligned_X = pd.DataFrame(
    {canonical: feature_df[raw_name] for canonical, raw_name in resolved_raw_by_canonical.items()},
    index=feature_df.index,
).copy()

# Track extra raw columns that are intentionally ignored by the canonical contract.
used_raw_columns = set(resolved_raw_by_canonical.values())
dropped_extra = sorted(set(feature_df.columns) - used_raw_columns)
missing_cols = []

# Reattach labels by position (not index label) to avoid accidental NaNs.
if label_col:
    labels_aligned = df[[label_col]].reset_index(drop=True)
    aligned_X = aligned_X.reset_index(drop=True)
    if len(aligned_X) != len(labels_aligned):
        raise ValueError(
            f"Feature/label row count mismatch after alignment: "
            f"X={len(aligned_X)}, y={len(labels_aligned)}"
        )
    df = pd.concat([aligned_X, labels_aligned], axis=1)
    print(f"Missing labels after reattach: {int(df[label_col].isna().sum())}")
else:
    df = aligned_X

print(f"Loaded shared feature list from {FEATURE_LIST_PATH}")
print(f"Loaded target feature alias map from {FEATURE_ALIAS_MAP_PATH}")
print(f"Aligned target feature count: {len(shared_features)}")
print(f"Dropped extra columns: {len(dropped_extra)}")
print(f"Missing required columns: {len(missing_cols)}")

Missing labels after reattach: 0
Loaded shared feature list from data/processed/shared_feature_space.json
Loaded target feature alias map from data/processed/target_feature_alias_map.json
Aligned target feature count: 68
Dropped extra columns: 12
Missing required columns: 0


In [5]:
# TEMP # Verify unique values for the canonical Label column after Attack -> Label transformation.
LABEL_CANDIDATES = ["Label", " Label", "label"]

label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find Label column. Tried {LABEL_CANDIDATES}")

values = df[label_col].astype("string").str.strip()
unique_values = sorted(values.dropna().unique().tolist())

print(f"Label column: {label_col}")
print(f"Unique label count: {len(unique_values)}")
print("Unique label values:")
for value in unique_values:
    print(f"- {value}")

Label column: Label
Unique label count: 10
Unique label values:
- Benign
- backdoor
- ddos
- dos
- injection
- mitm
- password
- ransomware
- scanning
- xss


In [6]:
### Label-space alignment ###
# (align labels according to predetermined shared label space)

SHARED_LABEL_SPACE_PATH = Path("data/processed/shared_label_space.json")
TARGET_LABEL_MAP_PATH = Path("data/processed/target_label_map.json")

if not SHARED_LABEL_SPACE_PATH.exists():
    raise FileNotFoundError(
        f"Shared label space file not found at {SHARED_LABEL_SPACE_PATH}"
    )
if not TARGET_LABEL_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Target label map file not found at {TARGET_LABEL_MAP_PATH}"
    )

with open(SHARED_LABEL_SPACE_PATH, "r", encoding="utf-8") as f:
    shared_label_space = json.load(f)
with open(TARGET_LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    target_label_map = json.load(f)

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Normalize raw labels for stable matching (trim spaces, keep missing as <NA>).
raw_labels = df[label_col].astype("string").str.strip()

# Detect genuinely missing labels (null/blank) separately from unmapped labels.
missing_label_mask = raw_labels.isna() | raw_labels.eq("")
missing_label_count = int(missing_label_mask.sum())

if missing_label_count > 0:
    missing_indices = df.index[missing_label_mask].tolist()
    preview_rows = missing_indices[:10]
    preview_values = [repr(v) for v in raw_labels.loc[preview_rows].tolist()]
    raise ValueError(
        f"Missing label values found at row indices {preview_rows} "
        f"(total={missing_label_count}). "
        f"Sample raw values at those rows: {preview_values}. "
        "Clean/drop these rows before label alignment."
    )

# Guardrail: mapping file must only map into allowed shared classes.
invalid_target_classes = sorted(
    set(target_label_map.values()) - set(shared_label_space)
)
if invalid_target_classes:
    raise ValueError(
        "target_label_map.json contains classes not present in shared_label_space.json: "
        f"{invalid_target_classes}"
    )

# Apply raw->shared mapping.
mapped_labels = raw_labels.map(target_label_map)

# Fail fast on unmapped non-missing raw labels to avoid silent label drift.
unmapped_mask = (~missing_label_mask) & mapped_labels.isna()
if unmapped_mask.any():
    unmapped_indices = df.index[unmapped_mask].tolist()
    unmapped_raw = raw_labels[unmapped_mask].tolist()
    unique_unmapped_raw = sorted(set(unmapped_raw))
    preview_pairs = list(zip(unmapped_indices, [repr(v) for v in unmapped_raw]))[:10]

    raise ValueError(
        f"Unmapped raw labels found: {[repr(v) for v in unique_unmapped_raw[:10]]} "
        f"(total unique={len(unique_unmapped_raw)}, total rows={len(unmapped_indices)}). "
        f"Sample row/value pairs: {preview_pairs}. "
        f"Update {TARGET_LABEL_MAP_PATH}."
    )

# Replace dataset labels with aligned shared classes.
df[label_col] = mapped_labels

# Sanity check: print class-frequency table after label alignment.
label_counts = df[label_col].value_counts(dropna=False).sort_values(ascending=False)
label_freq = (label_counts / len(df) * 100).round(2)

print("Label category frequencies after alignment:")
for cls in label_counts.index:
    print(f"- {cls}: {int(label_counts[cls])} ({label_freq[cls]:.2f}%)")

Label category frequencies after alignment:
- Benign: 2514059 (46.99%)
- Web Injection: 2427004 (45.36%)
- Brute Force: 340208 (6.36%)
- Probing: 36205 (0.68%)
- Malware: 32760 (0.61%)
- Denial of Service: 347 (0.01%)


In [7]:
### Train/Test Split ###
from sklearn.model_selection import train_test_split

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

X = df.drop(columns=[label_col]).copy()
y = df[label_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
    shuffle=True,
)

print(f"Train/Test sizes: {len(y_train)}/{len(y_test)}")

Train/Test sizes: 4280466/1070117


In [8]:
### Scaling (reuse scaler of source dataset) ###
# Note: exported data is already scaled

SCALER_PATH = Path("models/source_scaler.joblib")
if not SCALER_PATH.exists():
    raise FileNotFoundError(f"Source scaler not found at {SCALER_PATH}")

scaler = joblib.load(SCALER_PATH)

# Apply source-fitted scaler separately to target train/test split.
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames to preserve feature names/indexing.
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(f"Loaded scaler from {SCALER_PATH}")
print(f"Scaled splits shapes: train={X_train.shape}, test={X_test.shape}")

Loaded scaler from models/source_scaler.joblib
Scaled splits shapes: train=(4280466, 68), test=(1070117, 68)


In [9]:
### Label encoding (reuse encoder of shared label space) ###

ENCODER_PATH = Path("models/label_encoder.joblib")
if not ENCODER_PATH.exists():
    raise FileNotFoundError(f"Label encoder not found at {ENCODER_PATH}")

le = joblib.load(ENCODER_PATH)

# Ensure train/test labels are fully compatible with source-fitted encoder.
raw_train_labels = y_train.astype("string").str.strip()
raw_test_labels = y_test.astype("string").str.strip()
all_unknown = sorted((set(raw_train_labels.dropna()) | set(raw_test_labels.dropna())) - set(le.classes_))
if all_unknown:
    raise ValueError(
        f"Found labels not present in fitted source encoder: {all_unknown[:10]} "
        f"(total={len(all_unknown)})."
    )

y_train = pd.Series(le.transform(raw_train_labels), index=y_train.index, name="Label")
y_test = pd.Series(le.transform(raw_test_labels), index=y_test.index, name="Label")

print(f"Loaded label encoder from {ENCODER_PATH}")
print(f"Encoded classes ({len(le.classes_)}): {list(le.classes_)}")

Loaded label encoder from models/label_encoder.joblib
Encoded classes (6): ['Benign', 'Brute Force', 'Denial of Service', 'Malware', 'Probing', 'Web Injection']


In [10]:
### Calculate and export covariance and mean statistics ###
# Note: calculated from target training split only.

# Load shared feature space contract.
shared_feature_space_path = Path("data/processed/shared_feature_space.json")
if not shared_feature_space_path.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")

# Reuse unified parser so feature-space JSON is handled consistently across cells.
feature_order = load_feature_order(shared_feature_space_path)

# Ensure exact feature order before CORAL statistics.
X_target_train_aligned = X_train[feature_order]

# Convert to numpy for CORAL math.
X_trg = X_target_train_aligned.to_numpy(dtype=np.float32)

# 1) Feature-wise mean vector.
target_feature_mean = np.mean(X_trg, axis=0)

# 2) Centered target training data.
X_trg_centered = X_trg - target_feature_mean

# 3) Covariance matrix (core CORAL statistic).
target_covariance = np.cov(X_trg_centered, rowvar=False)

# Sanity checks.
assert target_covariance.shape[0] == target_covariance.shape[1], "Covariance matrix must be square"
assert target_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

# Package CORAL statistics.
coral_target_stats = {
    "feature_order": feature_order,
    "mean": target_feature_mean,
    "covariance": target_covariance,
}

# Persist for downstream domain adaptation pipeline.
coral_stats_path = Path("models/coral_target_stats.joblib")
coral_stats_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(coral_target_stats, coral_stats_path)

print("CORAL target statistics extracted and saved successfully.")
print("Verified: CORAL stats were computed from target train split only.")
print(f"Train rows used for CORAL: {len(X_target_train_aligned)}")
print(f"Saved to: {coral_stats_path}")
print(f"Features: {len(feature_order)}")
print(f"Covariance shape: {target_covariance.shape}")

CORAL target statistics extracted and saved successfully.
Verified: CORAL stats were computed from target train split only.
Train rows used for CORAL: 4280466
Saved to: models/coral_target_stats.joblib
Features: 68
Covariance shape: (68, 68)


In [11]:
### Export processed data ###

# Create output directory for processed target train/test splits.
output_dir = Path("data/processed/target")
output_dir.mkdir(parents=True, exist_ok=True)

# Save processed target train data.
train_df = pd.DataFrame(X_train, columns=X_train.columns)
train_df["Label"] = y_train.values
train_df.to_csv(output_dir / "train.csv", index=False)

# Save processed target test data.
test_df = pd.DataFrame(X_test, columns=X_test.columns)
test_df["Label"] = y_test.values
test_df.to_csv(output_dir / "test.csv", index=False)

print("Saved datasets:")
print(f"  Train: {len(train_df)} samples")
print(f"  Test: {len(test_df)} samples")
print(f"  Output directory: {output_dir}")

Saved datasets:
  Train: 4280466 samples
  Test: 1070117 samples
  Output directory: data/processed/target
